# v5: Per-vehicle repair-cost model

**Version:** v5  
**Kaggle submission:** **#5** `sub05-2026-11-20-v5_raw` → `outputs/submission_v5_raw.csv` (26,060.57) and **#6** `sub06-2026-11-20-v5_platt` → `outputs/submission_v5_platt.csv` (22,968.55). Isotonic variant built, not uploaded.

**Hypothesis:** v3/v4 use one repair-cost distribution for every vehicle. If repair cost is predictable from the sensors, a per-vehicle cost distribution improves **both** decisions: the budget (per-vehicle newsvendor quantile) and the flag (per-vehicle expected cost).

*Self-contained: needs only `../data/train.csv` and `../data/test.csv`. Part of the Fleet Maintenance project (Kaggle `mlprojectfleet2`); the cost, decision rule and validation protocol are derived in `fleet_maintenance.ipynb`.*

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, xgboost as xgb
from pathlib import Path
from scipy.stats import ks_2samp
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, r2_score, brier_score_loss, log_loss
from sklearn.model_selection import StratifiedKFold, KFold, train_test_split, cross_val_predict

DATA, OUT = Path("../data"), Path("outputs"); OUT.mkdir(exist_ok=True)
SEED = 42
INSPECT, UNDER, OVER = 10.0, 0.20, 0.02          # inspection cost; shortage cost / unit; unused-budget cost / unit
Q_STAR = UNDER / (UNDER + OVER)                  # newsvendor critical ratio 0.909
HISTB = [f"s{i:03d}" for i in range(54, 64)]     # 10-bin histogram that is shifted by one bin in test

def load(raw=False):
    """train/test with the v1 fixes: realign s054-s063 (test bin k+1 == train bin k), drop s047 (all NaN in
    test) and s146 (constant). raw=True returns the files untouched."""
    tr, te = pd.read_csv(DATA / "train.csv"), pd.read_csv(DATA / "test.csv")
    if not raw:
        te[HISTB[:-1]] = te[HISTB[1:]].to_numpy(); te[HISTB[-1]] = np.nan
        tr, te = tr.drop(columns=["s047", "s146"]), te.drop(columns=["s047", "s146"])
    return tr, te, [c for c in tr.columns if c.startswith("s")]

tr, te, S = load()
dev_idx, ho_idx = train_test_split(np.arange(len(tr)), test_size=0.2, stratify=tr.failure, random_state=SEED)
dev, ho = tr.iloc[dev_idx].reset_index(drop=True), tr.iloc[ho_idx].reset_index(drop=True)   # ho: locked 20% hold-out
y, c = dev.failure.to_numpy(), dev.repair_cost.fillna(0).to_numpy() * dev.failure.to_numpy()
y_ho, c_ho = ho.failure.to_numpy(), ho.repair_cost.fillna(0).to_numpy() * ho.failure.to_numpy()
c_all = tr.repair_cost.fillna(0).to_numpy() * tr.failure.to_numpy()
F = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y))   # the 5 fixed folds
print(f"train {tr.shape}  test {te.shape}  sensors {len(S)}  dev {len(dev)} / hold-out {len(ho)}  failures {int(y.sum())}")

train (56000, 172)  test (20000, 169)  sensors 168  dev 44800 / hold-out 11200  failures 318


In [2]:
def vehicle_cost(c, flag, b):
    return np.where(flag == 1, INSPECT + UNDER * np.maximum(c - b, 0) + OVER * np.maximum(b - c, 0), c)

def per_vehicle(c, flag, b):
    return float(vehicle_cost(np.asarray(c, float), np.asarray(flag), np.asarray(b, float)).mean())

fold_cost = lambda flag, b: np.array([per_vehicle(c[v], flag[v], b[v]) for _, v in F])

def write_submission(ids, flag, budget, name):
    flag, budget = np.asarray(flag).astype(int), np.maximum(np.asarray(budget, float), 0)
    assert len(ids) == 20000 and ids.is_unique and set(flag) <= {0, 1} and np.isfinite(budget).all()
    pd.DataFrame({"id": ids, "flag": flag, "budget": budget.round(2)}).to_csv(OUT / f"submission_{name}.csv", index=False)
    print(f"wrote outputs/submission_{name}.csv: flagged {flag.mean():.2%} of test vehicles")

def decide_global(p, fail_costs):
    """Same repair-cost distribution F (empirical failure costs) for every vehicle."""
    p = np.clip(np.asarray(p, float), 0, 1); fc = np.sort(np.asarray(fail_costs, float)); mu = fc.mean()
    b = np.zeros_like(p); need = p > 1 - Q_STAR
    b[need] = np.quantile(fc, 1 - (1 - Q_STAR) / p[need])                       # newsvendor budget
    fail = (UNDER * np.maximum(fc[None, :] - b[:, None], 0) + OVER * np.maximum(b[:, None] - fc[None, :], 0)).mean(1)
    return ((INSPECT + p * fail + (1 - p) * OVER * b) < p * mu).astype(int), b   # flag iff cheaper than not flagging

def decide_pv(p, m, resid, fail_costs=None, K=99, scale=None):
    """Per-vehicle cost distribution: log c | fail ~ m_i + scale_i * residuals (out-of-fold residuals).
    fail_costs given -> budget from that GLOBAL distribution (ablation); scale -> per-vehicle spread (v9)."""
    p, m = np.clip(np.asarray(p, float), 0, 1), np.asarray(m, float)
    sc = np.ones_like(m) if scale is None else np.asarray(scale, float)
    M = np.exp(m[:, None] + sc[:, None] * np.quantile(resid, (np.arange(K) + 0.5) / K)[None, :]); mu = M.mean(1)
    b = np.zeros_like(p); need = p > 1 - Q_STAR; level = 1 - (1 - Q_STAR) / p[need]
    b[need] = (np.exp(m[need] + sc[need] * np.quantile(resid, level)) if fail_costs is None
               else np.quantile(np.asarray(fail_costs, float), level))
    fail = (UNDER * np.maximum(M - b[:, None], 0) + OVER * np.maximum(b[:, None] - M, 0)).mean(1)
    return ((INSPECT + p * fail + (1 - p) * OVER * b) < p * mu).astype(int), b

class Platt:
    def fit(self, p, y):
        self.m = LogisticRegression(C=1e6, max_iter=1000).fit(self._x(p), y); return self
    def predict(self, p): return self.m.predict_proba(self._x(p))[:, 1]
    @staticmethod
    def _x(p): p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))[:, None]

class Iso:
    def fit(self, p, y):
        self.m = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(p, y); return self
    def predict(self, p): return self.m.predict(p)

make_cal = lambda kind: {"platt": Platt, "iso": Iso}[kind]()

PARAMS = dict(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
              min_child_weight=1, n_jobs=-1, random_state=SEED)                         # failure classifier (v2..)
CM = dict(n_estimators=200, max_depth=3, learning_rate=0.03, subsample=0.8, colsample_bytree=0.5,
          min_child_weight=5, n_jobs=-1, random_state=SEED)                             # repair-cost model (v5)
inner = KFold(5, shuffle=True, random_state=SEED)

def cost_model(params, Xa, lc, Xb):
    """XGBoost on log repair cost (failures only). Returns predictions for Xb and OUT-OF-FOLD residuals of Xa."""
    resid = lc - cross_val_predict(xgb.XGBRegressor(**params), Xa, lc, cv=inner)
    return xgb.XGBRegressor(**params).fit(Xa, lc).predict(Xb), resid

def fit_oof(X, params=PARAMS):
    """5-fold out-of-fold failure probabilities on dev, and probabilities for test from a model fit on all train."""
    o = np.zeros(len(dev))
    for a, b in F:
        o[b] = xgb.XGBClassifier(**params).fit(X.iloc[a], y[a]).predict_proba(X.iloc[b])[:, 1]
    return o

shift = lambda p, k: k * p / (1 - p) / (1 + k * p / (1 - p))       # prior shift by odds factor k

### The cost model
XGBoost (depth 3, 200 trees) on **log(repair cost)**, trained on failing vehicles only. Each vehicle's cost distribution is `log c | fail ~ m_i + residuals`, where the residuals are **out-of-fold** (inner 5-fold CV) so the spread is honest, and the back-transform uses residual quantiles (no exp(mean) bias). Nested: inside each outer fold, the cost model and its residuals use only that fold's training failures.

Ablation per probability variant, paired by fold: **A** = global F for flag and budget (v3/v4); **B** = per-vehicle flag, global-F budget; **D** = per-vehicle flag and budget (full v5).

In [3]:
oof = fit_oof(dev[S])
p_te_raw = xgb.XGBClassifier(**PARAMS).fit(tr[S], tr.failure).predict_proba(te[S])[:, 1]
variants = ("raw", "platt", "iso")
cost = {(v, m): [] for v in variants for m in "ABD"}; cover, r2s = [], []
for a, b in F:
    fa, fb = a[y[a] == 1], b[y[b] == 1]
    m_b, resid = cost_model(CM, dev.loc[fa, S], np.log(c[fa]), dev.loc[b, S])
    fc = c[a][c[a] > 0]
    mf = m_b[y[b] == 1]
    r2s.append(r2_score(np.log(c[fb]), mf)); cover.append(np.mean(np.log(c[fb]) <= mf + np.quantile(resid, Q_STAR)))
    for v in variants:
        p = oof[b] if v == "raw" else make_cal(v).fit(oof[a], y[a]).predict(oof[b])
        for m, (fl, bu) in (("A", decide_global(p, fc)), ("B", decide_pv(p, m_b, resid, fc)), ("D", decide_pv(p, m_b, resid))):
            cost[(v, m)].append(per_vehicle(c[b], fl, bu))
print(f"cost model R2 on held-out failures (log scale): {np.round(r2s, 2)}  mean {np.mean(r2s):.2f}")
print(f"budget quantile coverage (target {Q_STAR:.3f}): {np.round(cover, 2)}  mean {np.mean(cover):.3f}\n")
print("variant rule  CV cost/veh        paired delta vs A (per fold)             folds better")
for v in variants:
    A = np.array(cost[(v, "A")])
    for m in "ABD":
        x = np.array(cost[(v, m)]); d = x - A
        print(f"{v:6s}  {m}    {x.mean():.4f}±{x.std():.3f}   {str(np.round(d, 3)):40s} {int((d < 0).sum())}/5")

cost model R2 on held-out failures (log scale): [0.69 0.7  0.68 0.71 0.6 ]  mean 0.68
budget quantile coverage (target 0.909): [0.9  0.94 0.95 0.92 0.86]  mean 0.915

variant rule  CV cost/veh        paired delta vs A (per fold)             folds better
raw     A    1.2419±0.387   [0. 0. 0. 0. 0.]                         0/5
raw     B    1.0809±0.323   [-0.15   0.023 -0.191 -0.126 -0.362]     4/5
raw     D    0.9986±0.316   [-0.206 -0.103 -0.262 -0.205 -0.44 ]     5/5
platt   A    1.1073±0.220   [0. 0. 0. 0. 0.]                         0/5
platt   B    0.9938±0.244   [-0.166  0.054 -0.246 -0.08  -0.13 ]     4/5
platt   D    0.9037±0.235   [-0.236 -0.086 -0.313 -0.168 -0.216]     5/5
iso     A    1.0523±0.258   [0. 0. 0. 0. 0.]                         0/5
iso     B    1.0161±0.250   [ 0.013 -0.03  -0.066 -0.025 -0.074]     4/5
iso     D    0.9322±0.230   [-0.061 -0.166 -0.108 -0.098 -0.167]     5/5


### Submissions: cost model on all training failures; calibrators on all dev OOF predictions

In [4]:
f_all = tr[tr.failure == 1]
m_te, resid_all = cost_model(CM, f_all[S], np.log(f_all.repair_cost.to_numpy()), te[S])
print(f"test predicted failure cost: median {np.median(np.exp(m_te)):.0f}   (train failures: median {f_all.repair_cost.median():.0f})")
for v in variants:
    pt = p_te_raw if v == "raw" else make_cal(v).fit(oof, y).predict(p_te_raw)
    fl, bu = decide_pv(pt, m_te, resid_all); print(f"{v}: ", end=""); write_submission(te.id, fl, bu, f"v5_{v}")

test predicted failure cost: median 173   (train failures: median 176)
raw: wrote outputs/submission_v5_raw.csv: flagged 4.83% of test vehicles
platt: wrote outputs/submission_v5_platt.csv: flagged 6.29% of test vehicles


iso: wrote outputs/submission_v5_iso.csv: flagged 6.75% of test vehicles


## Results and caveats

| p | A (global F) | B (per-vehicle flag) | **D (full v5)** | D−A | folds better | Public |
|---|---|---|---|---|---|---|
| raw | 1.242 | 1.081 | 0.999 ± 0.316 | −0.243 | 5/5 | 26,061 (#5) |
| **Platt** | 1.107 | 0.994 | **0.904 ± 0.235** | −0.203 | 5/5 | **22,969 (#6)** |
| isotonic | 1.052 | 1.016 | 0.932 ± 0.230 | −0.120 | 5/5 | not uploaded |

- **Repair cost is predictable from the sensors:** R² **0.68** on log scale on held-out failures (0.60-0.71 per fold; RMSE 0.95 vs 1.62 for a constant), driven mostly by `s080-s085`. It is unrelated to the failure score (corr −0.08) and to follow-up (0.05).
- **The budget does what it should:** the 0.909 quantile covers **91.5%** of held-out failures (target 90.9%).
- **Most of the gain is the flag decision** (A→B), the rest the budget (B→D, ~0.08-0.09). Calibration still helps on top (raw 0.999 → Platt 0.904), and on the public set (26,061 vs 22,969).
- Cost-driving sensors do not drift between depots (KS 0.002-0.009), so the cost model transfers; the depot shift is in failure *probability*.

**Caveats:** only 318 failures train the cost model per fold; repair costs are **simulated**, which probably explains the high R² (a real process would be noisier); residuals are assumed equally spread for all vehicles (tested in v9: no gain from modelling it); the failure probability is still the larger source of error.